# FC Matrix Analysis — rest and task (behavioral) connectomes

Compare HCP1200 xcp-d functional connectomes across conditions: rest (4 runs), rest per session (`rest_S1`, `rest_S2`, the test-retest reference) and the seven HCP tasks (combined LR+RL runs). Caches: `Conn2Conn_data/fc/` (see its `README.md`); loaded through `HCP_Base(fc_conditions=...)`, so subjects and train/val/test partitions match the modeling pipeline. Plotting helpers live in `data.data_viz`.

1. Condition connectomes — population aggregates and single subjects (raw / demeaned)
2. Condition × condition similarity — edge correlation, Euclidean distance, geodesic distance
3. Subject × condition correlation structure — within- vs between-subject, raw and demeaned, per partition

Timepoints per subject (combined): rest 4800 · wm 810 · language 632 · motor 568 · social 548 · gambling 506 · relational 464 · emotion 352. Task FC is noisier than rest by different amounts, and single-subject task matrices are singular when TRs < parcels, so subject-level geodesic distances use shrinkage (1-λ)C + λI (default λ = 0.1).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "main.py").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data.dataset_utils as dataset_utils
import data.hcp_dataset as hcp_dataset
import data.data_viz as data_viz
importlib.reload(dataset_utils)
importlib.reload(hcp_dataset)
importlib.reload(data_viz)

from data.dataset_utils import FC_TASK_CONDITIONS
from data.hcp_dataset import HCP_Base
from data.data_viz import (
    plot_condition_connectomes,
    compute_condition_similarity,
    plot_condition_similarity,
    compute_subject_condition_correlations,
    plot_subject_condition_corrmap,
)

In [ ]:
PARCELLATION = 'Glasser'  # or '4S456Parcels'
SHUFFLE_SEED = 0
TASKS = list(FC_TASK_CONDITIONS)                  # emotion gambling language motor relational social wm
CONDITIONS = ['rest', 'rest_S1', 'rest_S2', *TASKS]

SAVE_FIGURES = False
FIG_DIR = REPO_ROOT / 'results/figures/fc_conditions'

base = HCP_Base(
    parcellation=PARCELLATION,
    shuffle_seed=SHUFFLE_SEED,
    source='SC',
    target='FC',
    data_load_mode='precomputed',
    expose_fc_sessions=True,          # rest_S1 / rest_S2
    fc_conditions=['rest', *TASKS],   # task caches; subjects = intersection with SC, FC, sessions, metadata
)

print(f'Parcellation: {PARCELLATION} | subjects with every condition: {len(base.metadata_df)}')
for part in ('train', 'val', 'test'):
    print(f'  {part}: {len(base.trainvaltest_partition_indices[part])}')

In [ ]:
def save(fig, name):
    if SAVE_FIGURES and fig is not None:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / f'{name}_{PARCELLATION}.png', dpi=200, bbox_inches='tight')

## 1. Condition connectomes

Population aggregates (shared [-1, 1] scale). `apply_fisher_z=True` averages in Fisher-z space.

In [ ]:
fig, axes, pop_mean = plot_condition_connectomes(base, CONDITIONS, partition='val', aggregate='mean', ncols=5)
save(fig, 'conn_val_mean')

fig, axes, pop_fz = plot_condition_connectomes(base, CONDITIONS, partition='val', aggregate='mean', apply_fisher_z=True, ncols=5)
save(fig, 'conn_val_fisherz_mean')

Partition means minus the train mean of the same condition (sanity check: should be small and unstructured).

In [ ]:
fig, axes, val_offset = plot_condition_connectomes(base, CONDITIONS, partition='val', demean_partition='train', ncols=5)

### Single subjects (raw and minus each condition's train mean)

In [ ]:
SUBJ_POSITION = 0  # position within the partition; or pass subject_id=...

fig, axes, subj_raw = plot_condition_connectomes(
    base, CONDITIONS, partition='val', position=SUBJ_POSITION, header_metadata='age_sex', ncols=5,
)
fig, axes, subj_dm = plot_condition_connectomes(
    base, CONDITIONS, partition='val', position=SUBJ_POSITION, demean_partition='train', header_metadata='age_sex', ncols=5,
)
print(f"shared residual vmax: {subj_dm['vmax']:.3f}")

## 2. Condition × condition similarity

Population level: between condition means of the partition (no shrinkage needed — means are full rank).

In [ ]:
sim_pop = compute_condition_similarity(base, CONDITIONS, partition='val', mode='population')
fig, axes = plot_condition_similarity(base, sim_pop)
save(fig, 'similarity_val_population')

pd.DataFrame(sim_pop['metrics']['pearson']['mean'], index=CONDITIONS, columns=CONDITIONS).round(3)

One subject (geodesic on shrunk matrices, λ shown in the title).

In [ ]:
sim_subj = compute_condition_similarity(base, CONDITIONS, partition='val', mode='subject', position=SUBJ_POSITION)
fig, axes = plot_condition_similarity(base, sim_subj)

Per-subject similarity averaged over the partition (annotated mean ± std). Affine-invariant geodesic costs a few seconds per subject; use `geodesic='log_euclidean'` or `max_subjects` for large partitions.

In [ ]:
sim_subjects = compute_condition_similarity(base, CONDITIONS, partition='val', mode='subjects', max_subjects=None)
fig, axes = plot_condition_similarity(base, sim_subjects, show_std=True)
save(fig, 'similarity_val_subjects')

## 3. Subject × condition correlation structure

Pearson r between every (subject, condition) edge vector. Blocks along the diagonal are within-subject cross-condition correlations; off-diagonal blocks are between-subject. *Demeaned* subtracts each condition's train mean (`demean='pooled'` subtracts one shared mean and keeps condition offsets). Subject ordering: `'original'`, `'family'`, `'demographic'`, `'age'`.

In [ ]:
fig, pair_fig, corr_val = plot_subject_condition_corrmap(base, CONDITIONS, partition='val', order_by='original')
save(fig, 'subjxcond_val'); save(pair_fig, 'subjxcond_val_pairs')
corr_val['summary'].round(4)

In [ ]:
fig, pair_fig, corr_val_family = plot_subject_condition_corrmap(
    base, CONDITIONS, partition='val', order_by='family', show_pair_matrices=False,
)

Condition-major blocks (subjects within each condition block) — the same correlations, grouped to show condition structure.

In [ ]:
fig, pair_fig, corr_val_cond = plot_subject_condition_corrmap(
    base, CONDITIONS, partition='val', block_by='condition', show_pair_matrices=False,
)

### Per partition

In [ ]:
PARTITION_MAX_SUBJECTS = {'train': 200, 'val': None, 'test': None}  # train is large: SK x SK grows quadratically

partition_summaries = {}
for part, n_max in PARTITION_MAX_SUBJECTS.items():
    fig, pair_fig, res = plot_subject_condition_corrmap(
        base, CONDITIONS, partition=part, max_subjects=n_max, show_pair_matrices=True,
    )
    save(fig, f'subjxcond_{part}'); save(pair_fig, f'subjxcond_{part}_pairs')
    partition_summaries[part] = res['summary']

partition_summary_table = pd.concat(partition_summaries, axis=1).round(4)
partition_summary_table

## Follow-ups

- Cross-condition fingerprinting: top-1 identification accuracy and differential identifiability (I_self − I_others) per condition pair.
- Network view: parcels ordered by network with boundaries, network-block means per condition, task − rest differences.